# Analysis of postsynaptic events

This tutorial shows how to analyze postsynaptic currents using the **[FindPeaks](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.find_peaks.html)** functions of scipy. To read the full tutorial, please see [Patch-clamp data analysis in Python: bursts of action potentials](https://spikesandbursts.wordpress.com/2022/07/03/patch-clamp-data-analysis-in-python-postsynaptic-currents-and-potentials/) of the [Spikes and Bursts](https://spikesandbursts.wordpress.com/) blog.

**References**
* [Find_Peaks from Scipy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.find_peaks.html#scipy.signal.find_peaks)




# Import the libraries

In [ ]:
import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib import gridspec

# Open pCLAMP abf files
import pyabf

# Scipy
import scipy
from scipy import signal
from scipy.signal import find_peaks
from scipy.optimize import curve_fit

# Interactive plots in Jupyter lab 
%matplotlib widget
plt.close('all')

# Create the paths

In [ ]:
notebook_name = 'postsynaptic_events'

# Data path to 'Data_example' folders. Change accordingly to your data structure.
data_path = os.path.dirname(os.getcwd())  # Moves one level up from the current directory

# Change the folder names accordingly
paths = {'data':  f'{data_path}/Data',
         'processed_data': f'{data_path}/Processed_data/{notebook_name}',
         'analysis': f'{data_path}/Analysis/{notebook_name}'}

# Make folders if they do not exist yet
for path in paths.values():
    os.makedirs(path, exist_ok=True)

# Load the data

Example data for this notebook in GitHub's data folder:
* ABF file: **pfc_sst_epscs.abf**, **pfc_sst_epscs.csv**

In [ ]:
# ABF files
filename = 'pfc_sst_epscs'

abf = pyabf.ABF(f"{paths['data']}/{filename}.abf")
print(abf)

# In case you have several sweeps/channels, select the right one
abf.setSweep(sweepNumber=0, channel=0) 

# Define the variables and sampling frequency
time = abf.sweepX  # in seconds
current = abf.sweepY
fs = int(abf.dataPointsPerMs *1000)

# Pre-processing: adjust baseline and filtering
* A basic baseline adjustement can be done by subtracting a fix value or the mean of part or the full trace.
* More information about Scipy filtering: https://docs.scipy.org/doc/scipy/reference/signal.html#filtering 

In [ ]:
# Define the signal variable: current or voltage
signal_raw = current

# Subtract mean or median of the full trace
signal_adjusted = signal_raw - np.median(signal_raw) 

# Option B: Subtract a fixed value
# signal_adjusted = signal_raw - 10 

# Option C: Subtract a percentage of values with low activity
# signal_adjusted = signal_raw - np.percentile(signal_raw, 25)

# Lowpass Bessel filter
b_lowpass, a_lowpass = signal.bessel(4,     # Order of the filter
                                     2000,  # Cutoff frequency
                                     'low', # Type of filter
                                     analog=False,  # Analog or digital filter
                                     norm='phase',  # Critical frequency normalization
                                     fs=fs)  # fs: sampling frequency

# Notch Besse filter (comment out if not needed)
b_notch, a_notch = signal.bessel(1,     # Order of the filter
                                 [59, 61],  # Cutoff frequency
                                 'bandstop', # Type of filter
                                 analog=False,  # Analog or digital filter
                                 fs=fs)  # fs: sampling frequency

# Combine both filters (comment out if not needed)
b_multiband = signal.convolve(b_lowpass, b_notch)
a_multiband = signal.convolve(a_lowpass, a_notch)

# For 2 filters (comment out if not needed)
signal_filtered = signal.filtfilt(b_multiband, a_multiband, signal_adjusted)


# Calculate standard deviation of the full trace
std_trace = np.std(signal_filtered)

# Calculate the 99th percentile for positive values and 10th percentile for negative values
baseline_positive_mean = np.percentile(signal_filtered[signal_filtered > 0], 99)
baseline_negative_mean = np.percentile(signal_filtered[signal_filtered < 0], 10)

# Create a baseline signal where values outside the range are set to NaN
baseline_signal = np.where((signal_filtered <= baseline_positive_mean) & (signal_filtered >= baseline_negative_mean), 
                           signal_filtered, np.nan)

# Calculate standard deviation of the baseline signal (ignoring NaN values)
std_baseline = np.nanstd(baseline_signal)

# Plot the raw trace
fig = plt.figure(figsize=(12, 6))
ax1 = fig.add_subplot(211)
ax1.set_title("Raw data")
ax1.plot(time, current, linewidth=0.5)

# Plot the filtered and adjusted trace
ax2 = fig.add_subplot(212, sharex=ax1, sharey=ax1)
ax2.set_title("Filtered data")
ax2.plot(time, signal_filtered, color='pink', linewidth=0.5)
ax2.plot(time, baseline_signal, color='gray', linewidth=0.5, alpha=0.5, label='Baseline')
ax2.legend()

# Add labels and legend
ax1.set_ylabel("Current (pA)")
ax2.set_ylabel("Current (pA)")
ax2.set_xlabel("Time (s)")
ax2.legend(frameon=False, handlelength=3, handleheight=0.7)

# Adjust layout and show the plot
fig.tight_layout()
plt.show()

# Print standard deviation results
print("Standard deviation of the full trace:", std_trace)
print("Mean negative value of the baseline:", baseline_negative_mean)
print("Standard deviation of the trace baseline:", std_baseline)

# Analyze the synaptic events


Two important considerations of the Find Peaks function:
* It only works for positive peaks so for inward currents or IPSPs, use the symbol '-' in front of the data variable to convert to possitive values.
* The FindPeaks function do not consider the time variable, only the recorded data points. To convert the values to time series, divide the results by the sampling rate (s) or by sampling rate/1000 (ms). 

**Parameters**. You can use 5 SD of baseline as reasoble starting point for event threshold. In most cases, [prominence](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.peak_prominences.html) is more useful because it detects how much a peak stands out from the sorrounded peaks.

In [ ]:
# Define the signal variable: current or voltage (raw or processed)
peaks_signal = signal_filtered  # trace after pre-processing 

# You could also select a range of the trace with the below option
# peaks_signal = current_filtered[5000:50000]

# Event window parameters
event_no = 1  # Event viewer: 0 is the first event
pretrigger_window = (1 * fs)/1000  # Pre-event time window in ms
posttrigger_window = (4 * fs)/1000  # Post-event time window in ms

# Set parameters of the Find peaks function
thresh_min = 8
thresh_max = 120
thresh_prominence = 15
thresh_min_width = 0.5 * (fs/1000)

# Find peaks function
peaks, peaks_dict = find_peaks(-peaks_signal, 
           height=(thresh_min, thresh_max),  # Min and max thresholds to detect peaks.
           threshold=None,  # Min and max vertical distance to neighboring samples.
           distance=None,  # Min horizontal distance between peaks.
           prominence=thresh_prominence,  # Vertical distance between the peak and lowest contour line.
           width=thresh_min_width,  # Min required width (in bins). E.g. For 10Khz, 10 bins = 1 ms.
           wlen=None,  # Window length to calculate prominence.
           rel_height=0.5,  # Relative height at which the peak width is measured.
           plateau_size=None)
 
# Create table with results
table = pd.DataFrame(columns = ['event', 'peak_index', 
                                'peak_time_s',
                                'event_window_start', 'event_window_end',
                                'peak_amp', 'width_ms', 
                                'inst_freq', 'isi_s', 
                                'area', 'decay_tau_log', 
                                'decay_tau_exp', 'rise_tau_exp'])
 
table.event = np.arange(1, len(peaks) + 1)
table.peak_index = peaks
table.peak_time_s = peaks / fs  # Divided by fs to get s
table.event_window_start = peaks_dict['left_ips'] - pretrigger_window
table.event_window_end = peaks_dict['right_ips'] + posttrigger_window
table.peak_amp = peaks_dict['peak_heights']  # height parameter is needed
table.width_ms = peaks_dict['widths']/(fs/1000) # Width (ms) at half-height

# Additional parameters (remember to add the columns to the dataframe)
# table.rise_half_amp_ms = (peaks - peaks_dict['left_ips'])/(fs/1000) 
# table.decay_half_amp_ms = (peaks_dict['right_ips'] - peaks)/(fs/1000)
 
# Calculations based on the parameters above
table.inst_freq = np.insert((1 / (np.array(table.peak_index[1:]) -
                                  np.array(table.peak_index[:-1])) * fs), 
                            0, np.nan) 

table.isi_s = np.diff(peaks, axis=0, prepend=peaks[0]) / fs

for i, event in table.iterrows():
    
    # Event area as absolute value (abs)
    individual_event = peaks_signal[int(event.event_window_start) : int(event.event_window_end)]
    table.loc[i, 'area'] = abs(round(individual_event.sum()/(fs/1000), 2))  # pA * ms
    
    # Decay tau from logistic regression
    decay_tau = abs(peaks_signal[int(event.peak_index) : int(event.event_window_end)])
    decay_tau_log = np.log(decay_tau)
    decay_width = int(len(decay_tau))
    decay_width_array = list(range(0, decay_width))
    slope, _ = np.polyfit(decay_width_array, decay_tau_log, 1)
    tau = -1 / slope
    table.loc[i, 'decay_tau_log'] = tau/(fs/1000) 

    # Decay tau from monoexponential fitting
    decay_tau = peaks_signal[int(event.peak_index) : int(event.event_window_end)]
    decay_width = int(len(decay_tau))
    decay_width_array = list(range(0, decay_width))
    a_initial = 200
    b_initial = 0.1
    # popt: optimal values for the parameters, pcov: estimated covariance of popt
    popt, pcov = curve_fit(lambda t, a, b: a * np.exp(b * t), 
                           decay_width_array, decay_tau, 
                           p0=(a_initial, b_initial), 
                           maxfev=2000)  # maxfev: number of iterations
    a = popt[0]  
    b = popt[1]      
    table.loc[i, 'decay_tau_exp'] = abs((1/b)/(fs/1000))
    
    # Rise tau from monoexponential fitting
    rise_tau = peaks_signal[int(event.event_window_start):int(event.peak_index)]
    rise_width = int(len(rise_tau))
    rise_width_array = list(range(0, rise_width))
    a_initial = 200
    b_initial = 0.1
    # popt: optimal values for the parameters, pcov: estimated covariance of popt
    popt, pcov = curve_fit(lambda t, a, b: a * np.exp(b * t), 
                           rise_width_array, rise_tau, 
                           p0=(a_initial, b_initial), 
                           maxfev=2000)  # maxfev: number of iterations
    a = popt[0]  
    b = popt[1]      
    table.loc[i, 'rise_tau_exp'] = abs((1/b)/(fs/1000))

    
# Plotting
fig = plt.figure(figsize=(18,4))
gridspec = fig.add_gridspec(ncols=2, nrows=1, width_ratios=[2, 1])

# Plot 1: Detected events in the trace
ax1 = fig.add_subplot(gridspec[0])  # gridspec specifies the ratio between plots
ax1.set_title("Events detection")   
ax1.plot(peaks_signal)
ax1.plot(peaks, peaks_signal[peaks], "r.")
for i, txt in enumerate(table.event):
    ax1.annotate(table.event[i], (peaks[i], peaks_signal[peaks][i]))
ax1.set_xlabel("Time bin")
ax1.set_ylabel("Current (pA)")
# ax1.axes.set_xlim(4000, 10000)  # OptionaL: Zoom in the trace
 
# Plot 2: Event viewer
ax2 = fig.add_subplot(gridspec[1]) 
ax2.set_title("Event viewer")
ax2.plot(peaks_signal, "gray")
ax2.plot(peaks, peaks_signal[peaks], "r.")
ax2.set_xlabel("Time bin")
ax2.set_ylabel("Current (pA)")

# Event time window
ax2.set_xlim(table.event_window_start[event_no], table.event_window_end[event_no]) 

# Labeling the event
line, = ax2.plot(peaks, peaks_signal[peaks], "r.") 
line.set_label(table.event[event_no]) 
ax2.legend()

# Save table and plot
fig.savefig(f"{paths['analysis']}/{filename}_results.svg", dpi=300)
table.to_csv(f"{paths['analysis']}/{filename}_results.csv", index=False)
       
# Show graph and table
plt.show()
table.round(3)  # round: display of decimal numbers in the table

# Save the analysis parameters

In [ ]:
analysis_parameters = [{'sampling frequency': fs,
                        'baseline adjustment': 'average substraction',
                        'lowpass_filter': 'bessel',
                        'lowpass_freq': '2000',
                        'notch_filter': 'bessel',
                        'notch_freq': '60',
                        'thresh_min': thresh_min,
                        'thresh_max': thresh_max,
                        'thresh_prominence': thresh_prominence,
                        'thresh_min_widht': thresh_min_width}]                     

# Option A: Save to npy file
np.save(f"{paths['analysis']}/{filename}_params", analysis_parameters)

# Option B: Save to txt file
txt = open(f"{paths['analysis']}/{filename}_params.txt","w")
txt.write(str(analysis_parameters))
txt.close()

# Option C: Save to csv file
df = pd.DataFrame.from_dict(analysis_parameters) 
df.to_csv(f"{paths['analysis']}/{filename}_params.csv", index=False, header=True)

In [ ]:
# To open the params file

# npy file
# analysis_parameters_npy = np.load(f"{paths['analysis']}/{filename}_params.npy", 
#                                   allow_pickle=True).item()

# Read the txt file
# analysis_parameters_txt = open(f"{paths['analysis']}/{filename}_params.txt", "r")

# Read the csv file
analysis_parameters_csv = pd.read_csv(f"{paths['analysis']}/{filename}_params.csv")
analysis_parameters_csv

# Comparison of fitting methods for synaptic events

To visually compare the fitting between the log and mono-exponential methods.

In [ ]:
for i, event in table.iterrows():
    if 5 <= i <= 10:  # To limit the range of events shown. 
        
        # Log fitting
        decay_tau = abs(signal_filtered[int(event.peak_index):int(event.event_window_end)])
        log_decay_tau = np.log(decay_tau)
        decay_width = int(len(decay_tau))
        decay_width_array = list(range(0, decay_width))
        slope, intercept = np.polyfit(decay_width_array, log_decay_tau, 1)
        
        # Exponential fitting
        decay_tau = signal_filtered[int(event.peak_index) : int(event.event_window_end)]
        decay_width = int(len(decay_tau))
        decay_width_array = list(range(0, decay_width))
        a_initial = -200
        b_initial = 0.1
        # popt: optimal values for the parameters, pcov: estimated covariance of popt
        popt, pcov = curve_fit(lambda t, a, b: a * np.exp(b * t),
                               decay_width_array, decay_tau, p0=(a_initial, b_initial))
        a = popt[0]  # 
        b = popt[1]      
        x_fitted2 = np.linspace(np.min(decay_width_array), np.max(decay_width_array))
        y_fitted2 = a * np.exp(b * x_fitted2)
    
        # Plot the log fitting
        fig = plt.figure(figsize=(5, 2))
        a_log = np.exp(intercept)
        b_log = slope
        x_fitted = np.linspace(np.min(decay_width_array), np.max(decay_width_array), 100)
        y_fitted = a_log * np.exp(b_log * x_fitted)
        ax1 = fig.add_subplot(121)
        ax1.set_title("Log method") 
        ax1.plot(decay_width_array, decay_tau)
        ax1.plot(x_fitted, -y_fitted)
        ax1.set_ylabel("Current (pA)")
        
        # Plot the exponential fitting
        ax2 = fig.add_subplot(122)
        ax2.plot(decay_width_array, decay_tau)
        ax2.plot(x_fitted2, y_fitted2)
        ax2.set_title("Exponential method") 
        fig.tight_layout()
        plt.show()

plt.show()

# Statistics: cumulative histograms

This is a simple example of how to calculate two summary statistics and do cumulative histograms using the parameter Peak Amplitude. Replace all the 'table.Peak_Amp_pA' in the script for other parameter from the table (e.g. 'table.isi_s') to plot other variables. 

* [Kolmogorov-Smirnoff test](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.kstest.html)

In [ ]:
# Plot settings
fig = plt.figure(figsize=(10, 4))
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2)

# Histogram data (use other variable as needed)
hist_data = table.peak_amp

# Bin size
bin_size = 2 # Bin size for the histograms

# Calculate the number of bins required
num_bins = int(np.ceil((max(hist_data) - min(hist_data)) / bin_size))

# Plot the histogram
ax1.hist(hist_data, bins=num_bins, range=(min(hist_data), max(hist_data)))
ax1.set_title('Histogram')
ax1.set_ylabel("Count")
ax1.set_xlabel("Peak Amplitude (pA)")

# Cumulative histogram
ax2.hist(hist_data, bins=len(np.unique(hist_data)),  # Length of unique values
         range=(min(hist_data), max(hist_data)),
         histtype = 'step',
         cumulative=True,
         density=True)

# Plot the cumulative histogram
ax2.set_title('Cumulative histogram')
ax2.set_ylabel("Cumulative probability")
ax2.set_xlabel("Peak Amplitude (pA)")
fig.tight_layout()


# Summary statistics
events = len(table.event)
time_s = len(time) / fs
mean_frequency = events / time_s
mean_amplitude = np.average(hist_data)
std_amplitude = np.std(hist_data)

# Create a pandas DataFrame
summary_stats = pd.DataFrame({
    'events': [events],
    'duration': [time_s],
    'mean_frequency': [mean_frequency],
    'mean_amplitude': [mean_amplitude],
    'std_amplitude': [std_amplitude]
})

# Save table and plot
fig.savefig(f"{paths['analysis']}/{filename}_{hist_data.name}_hist.png", dpi=300)
summary_stats.to_csv(f"{paths['analysis']}/{filename}_{hist_data.name}_hist.csv", index=False)

plt.show()
summary_stats

# Analyze multiple sweeps from text file

In [ ]:
# Load the file
filename = "pfc_sst_epscs"
data_path = f"{paths['data']}/{filename}.csv" 
data = np.loadtxt(fname=data_path, delimiter = ",")


# Define the signals 
time = data[:, 0]
currents = data[:, 1:3] # Select the number of traces you have\

peaks_signals = currents

# Plot the sweeps
plt.figure(figsize=(10,3))
for i, current in enumerate(currents.T):
    plt.plot(time/1000, current, label=f'sweep {i+1}')
plt.xlabel('Time (s)')
plt.ylabel('I (pA)')
plt.legend()
plt.show()

# Sampling rate
fs = 10000

# Event window parameters
pretrigger_window = (2 * fs)/1000  # Pre-event time window in ms
posttrigger_window = (5 * fs)/1000  # Post-event time window in ms
event_no = 0  # Event viewer: 0 is the first event

# Set parameters of the Find peaks function
thresh_min = 10
thresh_max = 120
thresh_prominence = 16
thresh_min_width = 0.9 * (fs/1000)

# Empty list to store the tables for each trace
table_list = []

# Loop trough the current columns
for i, peaks_signal in enumerate(peaks_signals.T):
    peaks, peaks_dict = find_peaks(-peaks_signal, 
                                   height=(thresh_min, thresh_max),  
                                   threshold=None, 
                                   distance=None, 
                                   prominence=thresh_prominence,  
                                   width=thresh_min_width, 
                                   wlen=None, 
                                   rel_height=0.5, 
                                   plateau_size=None)
 
    # Create table with results
    table = pd.DataFrame(columns = ['trace_index', 'event', 'peak_index', 
                                    'peak_time_s',
                                    'event_window_start', 'event_window_end',
                                    'peak_amp', 'width_ms', 
                                    'inst_freq', 'isi_s', 
                                    'area'])
    
    table.event = np.arange(1, len(peaks) + 1)
    
    table.trace_index = i  # Add index for each sweep
    
    table.peak_index = peaks
    table.peak_time_s = peaks / fs  # Divided by fs to get s
    table.event_window_start = peaks_dict['left_ips'] - pretrigger_window
    table.event_window_end = peaks_dict['right_ips'] + posttrigger_window
    table.peak_amp = peaks_dict['peak_heights']  # height parameter is needed
    table.width_ms = peaks_dict['widths']/(fs/1000) # Width (ms) at half-height
    
    # Calculations based on the parameters above
    table.inst_freq = np.insert((1 / (np.array(table.peak_index[1:]) -
                                      np.array(peaks_dict['left_ips'][:-1])) * fs), 
                                0, np.nan)
    
    table.isi_s = np.diff(peaks, axis=0, prepend=peaks[0]) / fs
 
    # Area
    for i, event in table.iterrows():
        individual_event = peaks_signal[int(event.event_window_start) : int(event.event_window_end)]
        table.loc[i, 'area'] = np.round(individual_event.sum(), 1)/(fs/1000) 

    # Append the table to the list
    table_list.append(table)

# Concatenate the tables in the list
table = pd.concat(table_list, ignore_index=True)

# Save the table
table.to_csv(f"{paths['analysis']}/{filename}_csv_epscs_results.csv", index=False)

table